# DiT Fig.2 Generalization Sweep: Results Check

Clean readout for the Transformer/DiT version of the CAMELS HI memorization-to-generalization sweep.

This notebook assumes the Great Lakes jobs have already produced train, sample, PCA, and SSCD outputs under `results/nf_generalize_fig2_dit/`. It is meant to answer three practical questions:

1. Did every DiT training/sampling/evaluation output land where expected?
2. Where does DiT move from training-slice copying to novel generation under PCA and SSCD nearest-neighbor tests?
3. How does the DiT transition compare with the existing UNet Fig.2 sweep, if those baseline tables are present?


## tl;dr

Run all cells on Great Lakes after the DiT sweep completes. The notebook will print:

- a file audit for the 10 DiT dataset sizes, `2^6` through `2^15`;
- PCA and SSCD generalization curves;
- an estimated `N50`, the training-set size where the generalization score crosses 0.5;
- an optional DiT-vs-UNet comparison if the original Fig.2 tables are present.

The `N50` number is a diagnostic, not a law: with one DiT architecture, it tells us where this model transitions, but not yet how Transformer capacity scales.


## Setup


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display


def find_project_dir() -> Path:
    env = os.environ.get('DIFFUSION_PROJECT_DIR')
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'scripts').exists() and (candidate / 'notebooks').exists():
            return candidate
    return here


PROJECT_DIR = find_project_dir()
SWEEP_NAME = 'nf_generalize_fig2_dit'
RESULTS_DIR = PROJECT_DIR / 'results' / SWEEP_NAME
TABLE_DIR = RESULTS_DIR / 'tables'
QUICKCHECK_DIR = RESULTS_DIR / 'quickcheck'
SAMPLE_DIR = RESULTS_DIR / 'samples'
MANIFEST_PATH = PROJECT_DIR / 'local' / SWEEP_NAME / 'manifest.json'
SAMPLE_LABEL = os.environ.get('SAMPLE_LABEL', 'dpm50')
SEED = int(os.environ.get('SEED', '123'))

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 300,
    'font.size': 14,
    'axes.labelsize': 15,
    'axes.titlesize': 16,
    'legend.fontsize': 12,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
})

print('PROJECT_DIR =', PROJECT_DIR)
print('RESULTS_DIR =', RESULTS_DIR)
print('SAMPLE_LABEL =', SAMPLE_LABEL)
print('SEED =', SEED)


## Context & Methods

- Model: DiT-base Transformer diffusion model on `128 x 128` CAMELS HI slices.
- Conditioning: this is the unconditional Fig.2-style sweep. The DiT implementation uses a single null class label internally because the diffusers DiT block requires `class_labels` for its adaLN path.
- Sweep: one run for each training-set size, `N_2D = 2^6, ..., 2^15`.
- Sampling: DPM-Solver, 50 steps, `512` generated slices per run.
- Diagnostics: nearest-neighbor similarity to real training slices in PCA and SSCD embedding spaces. High similarity means the generated sample is training-set-like; the plotted generalization score is high when generated fields are not unusually close to training slices.

Use this as a model-family comparison to the UNet Fig.2 sweep. It does not replace the full physical-fidelity checks.


## Data Audit


In [ ]:
def read_json(path: Path) -> Any | None:
    if not path.exists():
        return None
    with path.open() as f:
        return json.load(f)


def rel(path: Path) -> str:
    try:
        return str(path.relative_to(PROJECT_DIR))
    except ValueError:
        return str(path)


def dataset_tag_from_name(name: str) -> str | None:
    m = re.search(r'd2p(\d+)', str(name))
    if not m:
        return None
    return 'd2p' + m.group(1)


def dataset_size_from_tag(tag: str | None) -> int | None:
    if not tag:
        return None
    m = re.match(r'd2p(\d+)', tag)
    if not m:
        return None
    return 2 ** int(m.group(1))


def sample_path_for(row: pd.Series) -> Path:
    raw = str(row.get('sample_path', '') or '')
    if raw:
        raw = raw.format(seed=SEED, sample_label=SAMPLE_LABEL)
        path = Path(raw)
        return path if path.is_absolute() else PROJECT_DIR / path
    run_name = row.get('run_name') or row.get('name')
    tag = row.get('dataset_tag') or dataset_tag_from_name(str(run_name))
    if tag is None:
        return SAMPLE_DIR / f'unknown_seed{SEED}_{SAMPLE_LABEL}.npz'
    return SAMPLE_DIR / f'nf_fig2_dit_base_{tag}_noaug_200k_seed{SEED}_{SAMPLE_LABEL}.npz'


manifest_obj = read_json(MANIFEST_PATH)
if manifest_obj is None:
    display(Markdown(f'**Missing manifest:** `{rel(MANIFEST_PATH)}`'))
    manifest_df = pd.DataFrame()
else:
    rows = manifest_obj.get('runs', manifest_obj if isinstance(manifest_obj, list) else [])
    manifest_df = pd.DataFrame(rows)
    if 'run_name' not in manifest_df.columns and 'name' in manifest_df.columns:
        manifest_df['run_name'] = manifest_df['name']
    if 'dataset_tag' not in manifest_df.columns:
        manifest_df['dataset_tag'] = manifest_df['run_name'].map(dataset_tag_from_name)
    if 'dataset_size' not in manifest_df.columns:
        manifest_df['dataset_size'] = manifest_df['dataset_tag'].map(dataset_size_from_tag)
    manifest_df['sample_path_resolved'] = manifest_df.apply(sample_path_for, axis=1)
    manifest_df['sample_exists'] = manifest_df['sample_path_resolved'].map(Path.exists)
    manifest_df['sample_size_mb'] = manifest_df['sample_path_resolved'].map(lambda p: p.stat().st_size / 1024**2 if p.exists() else np.nan)

    show_cols = [c for c in [
        'run_name', 'dataset_tag', 'dataset_size', 'sample_exists', 'sample_size_mb',
        'config_path', 'output_dir', 'sample_path_resolved'
    ] if c in manifest_df.columns]
    display(manifest_df[show_cols].sort_values('dataset_size'))
    print(f"sample files present: {manifest_df['sample_exists'].sum()} / {len(manifest_df)}")

expected_tables = [
    TABLE_DIR / 'nf_generalize_fig2_dit_pca_full_nn_metrics.csv',
    TABLE_DIR / 'nf_generalize_fig2_dit_pca_full_nn_mode_norms.csv',
    TABLE_DIR / 'nf_generalize_fig2_dit_pca_full_nn_similarity_histograms.csv',
    TABLE_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_metrics.csv',
]
expected_figures = [
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_paper_style_gl_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_paper_style_gl_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_similarity_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_similarity_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_copy_fraction_curves.png',
    QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_copy_fraction_curves.png',
]

audit = pd.DataFrame({
    'path': [rel(p) for p in expected_tables + expected_figures],
    'kind': ['table'] * len(expected_tables) + ['figure'] * len(expected_figures),
    'exists': [p.exists() for p in expected_tables + expected_figures],
    'size_mb': [p.stat().st_size / 1024**2 if p.exists() else np.nan for p in expected_tables + expected_figures],
})
display(audit)


## Load Metrics


In [ ]:
def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if not path.exists():
        display(Markdown(f'**Missing table:** `{rel(path)}`'))
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f'loaded {rel(path)}: {len(df)} rows, {len(df.columns)} columns')
    return df


def add_generalization_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    df = df.copy()
    for q in ['q50', 'q68', 'q90', 'q95', 'q99']:
        copy_col = f'gen_copy_fraction_{q}'
        gl_col = f'gen_gl_{q}'
        if gl_col not in df.columns and copy_col in df.columns:
            df[gl_col] = 1.0 - df[copy_col]
    # Some older tables used shorter names.
    for q in ['q90', 'q95', 'q99']:
        if f'gen_gl_{q}' not in df.columns and f'copy_fraction_{q}' in df.columns:
            df[f'gen_gl_{q}'] = 1.0 - df[f'copy_fraction_{q}']
    if 'dataset_tag' not in df.columns:
        name_col = 'run_name' if 'run_name' in df.columns else df.columns[0]
        df['dataset_tag'] = df[name_col].map(dataset_tag_from_name)
    if 'dataset_size' not in df.columns:
        df['dataset_size'] = df['dataset_tag'].map(dataset_size_from_tag)
    return df


pca_metrics = add_generalization_columns(read_csv_if_exists(TABLE_DIR / 'nf_generalize_fig2_dit_pca_full_nn_metrics.csv'))
sscd_metrics = add_generalization_columns(read_csv_if_exists(TABLE_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_metrics.csv'))

for feature_name, df in [('PCA', pca_metrics), ('SSCD', sscd_metrics)]:
    display(Markdown(f'### {feature_name} metrics'))
    if df.empty:
        continue
    print('generalization columns:', [c for c in df.columns if c.startswith('gen_gl')])
    preferred = [
        'run_name', 'dataset_tag', 'dataset_size', 'n_generated', 'n_train',
        'gen_gl_q90', 'gen_gl_q95', 'gen_gl_q99',
        'gen_copy_fraction_q95', 'threshold_q95', 'gen_nn_median', 'gen_nn_q95'
    ]
    cols = [c for c in preferred if c in df.columns]
    display(df[cols].sort_values('dataset_size') if cols else df.head())


## DiT Generalization Curves

The main DiT readout is the generalization score versus training-set size. A score near zero means generated fields are often too close to training slices under the chosen embedding. A score near one means generated fields are not unusually close to the training set relative to the real-data baseline.


In [ ]:
def format_power_ticks(ax, values):
    vals = sorted({int(v) for v in values if pd.notna(v) and v > 0})
    if not vals:
        return
    ax.set_xscale('log', base=2)
    ax.set_xticks(vals)
    ax.set_xticklabels([rf'$2^{{{int(round(math.log2(v)))}}}$' for v in vals])


def plot_dit_generalization_curves(metrics_by_feature: dict[str, pd.DataFrame], quantile: str = 'q95') -> Path | None:
    gl_col = f'gen_gl_{quantile}'
    fig, ax = plt.subplots(figsize=(8.6, 5.4), constrained_layout=True)
    colors = {'PCA': '#0072B2', 'SSCD': '#D55E00'}
    markers = {'PCA': 'o', 'SSCD': 's'}
    all_x = []
    plotted = False

    for feature_name, df in metrics_by_feature.items():
        if df.empty or gl_col not in df.columns or 'dataset_size' not in df.columns:
            continue
        sub = df.dropna(subset=['dataset_size', gl_col]).sort_values('dataset_size')
        if sub.empty:
            continue
        all_x.extend(sub['dataset_size'].astype(float).tolist())
        ax.plot(
            sub['dataset_size'], sub[gl_col],
            marker=markers.get(feature_name, 'o'), ms=8, lw=3,
            color=colors.get(feature_name), label=feature_name,
        )
        plotted = True

    if not plotted:
        display(Markdown(f'No `{gl_col}` columns found to plot.'))
        plt.close(fig)
        return None

    format_power_ticks(ax, all_x)
    ax.axhline(0.5, color='0.35', lw=1.5, ls=':', label='0.5 transition marker')
    ax.set_ylim(-0.04, 1.04)
    ax.set_xlabel(r'Training set size $N_{2D}$')
    ax.set_ylabel('Generalization score')
    ax.set_title(f'DiT-base memorization-to-generalization check ({quantile})')
    ax.grid(True, alpha=0.22)
    ax.legend(frameon=False, loc='lower right')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

    QUICKCHECK_DIR.mkdir(parents=True, exist_ok=True)
    out = QUICKCHECK_DIR / f'nf_generalize_fig2_dit_combined_gl_curves_{quantile}.png'
    fig.savefig(out, bbox_inches='tight')
    plt.show()
    print('wrote', out)
    return out

combined_curve = plot_dit_generalization_curves({'PCA': pca_metrics, 'SSCD': sscd_metrics}, quantile='q95')


## Transition Summary

`N50` is the interpolated training-set size where the generalization score crosses 0.5. This is a compact way to compare transitions, but it should be read with the full curve because a single midpoint can hide changes in slope or tail behavior.


In [ ]:
def interpolate_crossing(df: pd.DataFrame, ycol: str, threshold: float = 0.5) -> dict[str, Any]:
    if df.empty or ycol not in df.columns or 'dataset_size' not in df.columns:
        return {'status': 'missing', 'n_cross': np.nan, 'log2_n_cross': np.nan}
    sub = df[['dataset_size', ycol]].dropna().sort_values('dataset_size')
    sub = sub[sub['dataset_size'] > 0]
    if sub.empty:
        return {'status': 'missing', 'n_cross': np.nan, 'log2_n_cross': np.nan}

    x = np.log2(sub['dataset_size'].astype(float).to_numpy())
    y = sub[ycol].astype(float).to_numpy()
    if y[0] >= threshold:
        return {'status': 'left_censored', 'n_cross': 2 ** x[0], 'log2_n_cross': x[0]}
    if y[-1] < threshold:
        return {'status': 'right_censored', 'n_cross': 2 ** x[-1], 'log2_n_cross': x[-1]}

    for i in range(len(y) - 1):
        y0, y1 = y[i], y[i + 1]
        if (y0 <= threshold <= y1) or (y1 <= threshold <= y0):
            if y1 == y0:
                xc = x[i]
            else:
                frac = (threshold - y0) / (y1 - y0)
                xc = x[i] + frac * (x[i + 1] - x[i])
            return {'status': 'interpolated', 'n_cross': 2 ** xc, 'log2_n_cross': xc}
    return {'status': 'not_found', 'n_cross': np.nan, 'log2_n_cross': np.nan}


rows = []
for feature_name, df in [('PCA', pca_metrics), ('SSCD', sscd_metrics)]:
    for q in ['q90', 'q95', 'q99']:
        col = f'gen_gl_{q}'
        result = interpolate_crossing(df, col, threshold=0.5)
        rows.append({
            'feature': feature_name,
            'score_col': col,
            'threshold': 0.5,
            **result,
        })

transition_df = pd.DataFrame(rows)
display(transition_df)

if len(transition_df):
    out = TABLE_DIR / 'nf_generalize_fig2_dit_transition_summary.csv'
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    transition_df.to_csv(out, index=False)
    print('wrote', out)


## Compare DiT With Existing UNet Sweep (Optional)

This section runs only if the original UNet Fig.2 PCA/SSCD tables are present in `results/nf_generalize_fig2/tables/`. It is useful for checking whether the Transformer transition is left-shifted, right-shifted, or similar to the UNet baselines.


In [ ]:
UNET_RESULTS_DIR = PROJECT_DIR / 'results' / 'nf_generalize_fig2'
UNET_TABLE_DIR = UNET_RESULTS_DIR / 'tables'

unet_pca = add_generalization_columns(read_csv_if_exists(UNET_TABLE_DIR / 'nf_generalize_fig2_pca_full_nn_metrics.csv'))
unet_sscd = add_generalization_columns(read_csv_if_exists(UNET_TABLE_DIR / 'nf_generalize_fig2_sscd_full_nn_metrics.csv'))


def arch_label(raw: Any) -> str:
    text = str(raw)
    mapping = {'u64': 'UNet-64', 'u128': 'UNet-128', 'u256': 'UNet-256', 'dit': 'DiT-base', 'dit_base': 'DiT-base'}
    return mapping.get(text, text)


def infer_arch_column(df: pd.DataFrame) -> pd.Series:
    if 'arch' in df.columns:
        return df['arch'].astype(str)
    if 'arch_label' in df.columns:
        return df['arch_label'].astype(str)
    if 'run_name' in df.columns:
        def from_name(name):
            m = re.search(r'(u64|u128|u256|dit_base|dit)', str(name))
            return m.group(1) if m else 'unknown'
        return df['run_name'].map(from_name)
    return pd.Series(['unknown'] * len(df), index=df.index)


def plot_dit_vs_unet(feature_name: str, dit_df: pd.DataFrame, unet_df: pd.DataFrame, quantile: str = 'q95') -> Path | None:
    gl_col = f'gen_gl_{quantile}'
    if dit_df.empty or gl_col not in dit_df.columns:
        display(Markdown(f'No DiT `{feature_name}` `{gl_col}` data.'))
        return None

    fig, ax = plt.subplots(figsize=(9.0, 5.6), constrained_layout=True)
    all_x = []
    colors = {'u64': '#009E73', 'u128': '#D55E00', 'u256': '#0072B2'}
    markers = {'u64': '^', 'u128': 'o', 'u256': 's'}

    if not unet_df.empty and gl_col in unet_df.columns:
        tmp = unet_df.copy()
        tmp['arch_for_plot'] = infer_arch_column(tmp)
        for arch in ['u64', 'u128', 'u256']:
            sub = tmp[tmp['arch_for_plot'].astype(str) == arch].dropna(subset=['dataset_size', gl_col]).sort_values('dataset_size')
            if sub.empty:
                continue
            all_x.extend(sub['dataset_size'].astype(float).tolist())
            ax.plot(
                sub['dataset_size'], sub[gl_col],
                color=colors[arch], marker=markers[arch], lw=2.2, ms=7,
                alpha=0.55, label=arch_label(arch),
            )

    sub = dit_df.dropna(subset=['dataset_size', gl_col]).sort_values('dataset_size')
    all_x.extend(sub['dataset_size'].astype(float).tolist())
    ax.plot(
        sub['dataset_size'], sub[gl_col],
        color='black', marker='D', lw=3.4, ms=8,
        label='DiT-base',
    )
    ax.axhline(0.5, color='0.35', lw=1.4, ls=':')
    format_power_ticks(ax, all_x)
    ax.set_ylim(-0.04, 1.04)
    ax.set_xlabel(r'Training set size $N_{2D}$')
    ax.set_ylabel('Generalization score')
    ax.set_title(f'{feature_name}: DiT-base compared with UNet baselines ({quantile})')
    ax.grid(True, alpha=0.22)
    ax.legend(frameon=False, ncol=2, loc='lower right')
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

    out = QUICKCHECK_DIR / f'nf_generalize_fig2_dit_vs_unet_{feature_name.lower()}_{quantile}.png'
    fig.savefig(out, bbox_inches='tight')
    plt.show()
    print('wrote', out)
    return out

_ = plot_dit_vs_unet('PCA', pca_metrics, unet_pca, quantile='q95')
_ = plot_dit_vs_unet('SSCD', sscd_metrics, unet_sscd, quantile='q95')


## Existing Quickcheck Figures


In [ ]:
def show_existing_figure(path: Path, title: str, width: int = 950) -> None:
    display(Markdown(f'### {title}'))
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        display(Markdown(f'Missing: `{rel(path)}`'))

show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_paper_style_gl_curves.png', 'PCA paper-style GL curves')
show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_paper_style_gl_curves.png', 'SSCD paper-style GL curves')
show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_similarity_curves.png', 'PCA similarity curves')
show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_similarity_curves.png', 'SSCD similarity curves')
show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_pca_full_nn_copy_fraction_curves.png', 'PCA copy-fraction curves')
show_existing_figure(QUICKCHECK_DIR / 'nf_generalize_fig2_dit_sscd_full_nn_copy_fraction_curves.png', 'SSCD copy-fraction curves')


## Sample File Sanity Check

This inspects array keys and shapes without doing the expensive nearest-neighbor analysis again.


In [ ]:
def inspect_npz(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {'exists': False}
    with np.load(path) as data:
        keys = list(data.files)
        first = keys[0] if keys else None
        arr = data[first] if first else None
        return {
            'exists': True,
            'keys': ', '.join(keys[:8]),
            'first_key': first,
            'shape': tuple(arr.shape) if arr is not None else None,
            'dtype': str(arr.dtype) if arr is not None else None,
            'size_mb': path.stat().st_size / 1024**2,
        }

if manifest_df.empty:
    display(Markdown('No manifest available, so sample files were not inspected.'))
else:
    rows = []
    for _, row in manifest_df.sort_values('dataset_size').iterrows():
        path = row['sample_path_resolved']
        info = inspect_npz(path)
        rows.append({
            'dataset_tag': row.get('dataset_tag'),
            'dataset_size': row.get('dataset_size'),
            'path': rel(path),
            **info,
        })
    sample_inspect_df = pd.DataFrame(rows)
    display(sample_inspect_df)


## Takeaways


In [ ]:
def best_transition_line(feature: str) -> str:
    if transition_df.empty:
        return f'- {feature}: transition table missing.'
    sub = transition_df[(transition_df['feature'] == feature) & (transition_df['score_col'] == 'gen_gl_q95')]
    if sub.empty:
        return f'- {feature}: q95 generalization column missing.'
    row = sub.iloc[0]
    if pd.isna(row['n_cross']):
        return f'- {feature}: N50 not available ({row["status"]}).'
    return f'- {feature}: q95 N50 = 2^{row["log2_n_cross"]:.2f} = {row["n_cross"]:.0f} 2D images ({row["status"]}).'

sample_ok = None if manifest_df.empty else int(manifest_df['sample_exists'].sum())
sample_total = None if manifest_df.empty else len(manifest_df)

lines = [
    '### Notebook summary',
]
if sample_ok is not None:
    lines.append(f'- Sample audit: {sample_ok}/{sample_total} DiT sample files found.')
lines.append(best_transition_line('PCA'))
lines.append(best_transition_line('SSCD'))
lines.extend([
    '- Read PCA and SSCD together. PCA is sensitive to low-dimensional variance; SSCD is a learned image-similarity embedding. Agreement is stronger evidence than either diagnostic alone.',
    '- Next check: compare these DiT curves against the UNet curves above. If DiT shifts the transition, then architecture matters beyond parameter count.',
])

display(Markdown('\n'.join(lines)))


## Great Lakes Rerun Command

From the repo root on Great Lakes:

```bash
cd /home/jiamingp/diffusion_models_repo
jupyter nbconvert --execute --to notebook --inplace notebooks/nf_generalize_fig2_dit_results.ipynb
```

If you are using the Jupyter web session, just open this notebook and run all cells. The heavy PCA/SSCD nearest-neighbor work should not rerun here; this notebook reads the completed CSV and PNG outputs.
